In [ ]:
import pandas as pd
import time
import xgboost as xgb
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# 1. LOAD THE NEW DATA
print("Loading v2 4-dimensional feature matrix...")
df = pd.read_parquet("../data/processed/model_ready_features_v2.parquet")

# Explicitly cast the string column to a categorical datatype for XGBoost
df['ecoregion'] = df['ecoregion'].astype('category')

# 2. ARCHITECTURE SETUP
df['target_presence'] = (df['scientific_name'] == 'Calypte anna').astype(int)

# Isolate all 4 features
X = df[['elevation_m', 'ecoregion', 'slope_deg', 'aspect_deg']]
y = df['target_presence']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Calculate exact imbalance ratio
neg_cases = y_train.value_counts()[0]
pos_cases = y_train.value_counts()[1]
imbalance_ratio = neg_cases / pos_cases

# Initialize GPU Classifier
clf_model = xgb.XGBClassifier(
    enable_categorical=True, 
    tree_method='hist', 
    device='cuda',             
    scale_pos_weight=imbalance_ratio, 
    random_state=42
)

# 3. TRAINING & EVALUATION
print("\nTraining 4-Feature Classification Model on GPU...")
start_time = time.time()
clf_model.fit(X_train, y_train)
print(f"Classifier trained in {time.time() - start_time:.2f} seconds.")

print("\n--- Classification Results ---")
# (Leaving X_test as a DataFrame so the categorical Ecoregion feature is recognized)
preds = clf_model.predict(X_test)
print(classification_report(y_test, preds, target_names=['Absent (0)', 'Present (1)']))

# 4. FEATURE IMPORTANCE PLOT
fig, ax = plt.subplots(figsize=(10, 6))
# We use 'gain' to see which feature improved the model's accuracy the most
xgb.plot_importance(clf_model, ax=ax, importance_type='gain', show_values=False)
plt.title("XGBoost Feature Importance (Gain) - Hummingbird Habitat")
plt.tight_layout()
plt.show()

In [ ]:
# Save the trained XGBoost model to disk
model_path = "../data/processed/hummingbird_habitat_model.json"
clf_model.save_model(model_path)
print(f"Model successfully saved to {model_path}")